In [1]:

!pip install -q scikit-fuzzy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.8/920.8 kB 19.4 MB/s eta 0:00:00


In [2]:

import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl

# Define input and output variables
attendance = ctrl.Antecedent(np.arange(0, 101, 1), 'attendance')
marks = ctrl.Antecedent(np.arange(0, 101, 1), 'marks')
performance = ctrl.Consequent(np.arange(0, 101, 1), 'performance')

# Membership functions for attendance
attendance['low'] = fuzz.trimf(attendance.universe, [0, 0, 50])
attendance['medium'] = fuzz.trimf(attendance.universe, [25, 50, 75])
attendance['high'] = fuzz.trimf(attendance.universe, [50, 100, 100])

# Membership functions for marks
marks['low'] = fuzz.trimf(marks.universe, [0, 0, 40])
marks['medium'] = fuzz.trimf(marks.universe, [30, 50, 70])
marks['high'] = fuzz.trimf(marks.universe, [60, 100, 100])

# Membership functions for performance
performance['poor'] = fuzz.trimf(performance.universe, [0, 0, 40])
performance['average'] = fuzz.trimf(performance.universe, [25, 50, 75])
performance['good'] = fuzz.trimf(performance.universe, [60, 100, 100])

print("Fuzzy variables and membership functions created.")


Fuzzy variables and membership functions created.


In [3]:

# Define fuzzy rules
rule1 = ctrl.Rule(marks['low'], performance['poor'])

rule2 = ctrl.Rule(
    marks['medium'] & attendance['low'],
    performance['poor']
)

rule3 = ctrl.Rule(
    marks['medium'] & attendance['medium'],
    performance['average']
)

rule4 = ctrl.Rule(
    marks['high'] & attendance['low'],
    performance['average']
)

rule5 = ctrl.Rule(
    marks['high'] & attendance['medium'],
    performance['good']
)

rule6 = ctrl.Rule(
    marks['high'] & attendance['high'],
    performance['good']
)

rule7 = ctrl.Rule(
    marks['medium'] & attendance['high'],
    performance['average']
)

# Create the control system
performance_ctrl = ctrl.ControlSystem([
    rule1, rule2, rule3, rule4, rule5, rule6, rule7
])

student_system = ctrl.ControlSystemSimulation(performance_ctrl)

# Test one student
student_system.input['attendance'] = 85
student_system.input['marks'] = 90

student_system.compute()

print("Attendance:", 85)
print("Marks:", 90)
print("Student Performance Score:", round(student_system.output['performance'], 2))


Attendance: 85
Marks: 90
Student Performance Score: 85.74


In [4]:

students = [
    (30, 25),
    (50, 55),
    (65, 75),
    (90, 95),
    (40, 85)
]

print("Attendance | Marks | Performance")
print("-" * 35)

for att, mark in students:
    student_system.input['attendance'] = att
    student_system.input['marks'] = mark
    student_system.compute()

    score = student_system.output['performance']

    print(f"{att:10} | {mark:5} | {score:.2f}")


Attendance | Marks | Performance
-----------------------------------
        30 |    25 | 16.54
        50 |    55 | 50.00
        65 |    75 | 83.46
        90 |    95 | 86.22
        40 |    85 | 73.22


In [5]:

# Subjects to schedule
subjects = ['AI', 'Maths', 'Physics']

# Available time slots
time_slots = ['9AM', '10AM', '11AM']

# Store the timetable
timetable = {}

# Check whether a time slot is already occupied
def is_valid(subject, slot):
    for scheduled_subject, scheduled_slot in timetable.items():
        if scheduled_slot == slot:
            return False
    return True

print("Subjects:", subjects)
print("Available time slots:", time_slots)
print("Constraint: No two subjects can share a time slot.")


Subjects: ['AI', 'Maths', 'Physics']
Available time slots: ['9AM', '10AM', '11AM']
Constraint: No two subjects can share a time slot.


In [6]:

def solve_timetable(index):
    # All subjects have been scheduled
    if index == len(subjects):
        return True

    subject = subjects[index]

    # Try each available time slot
    for slot in time_slots:
        if is_valid(subject, slot):
            timetable[subject] = slot

            # Schedule the next subject
            if solve_timetable(index + 1):
                return True

            # Backtrack if no solution is found
            del timetable[subject]

    return False


if solve_timetable(0):
    print("Timetable generated successfully:\n")
    for subject, slot in timetable.items():
        print(f"{subject}: {slot}")
else:
    print("No valid timetable exists.")


Timetable generated successfully:

AI: 9AM
Maths: 10AM
Physics: 11AM


In [7]:

# Modify the time slots to include a subject-specific constraint
timetable = {}

def is_valid(subject, slot):
    # AI must be scheduled at 10AM
    if subject == 'AI' and slot != '10AM':
        return False

    # No two subjects can share the same slot
    for scheduled_subject, scheduled_slot in timetable.items():
        if scheduled_slot == slot:
            return False

    return True


if solve_timetable(0):
    print("Updated timetable:\n")
    for subject, slot in timetable.items():
        print(f"{subject}: {slot}")
else:
    print("No valid timetable exists.")


Updated timetable:

AI: 10AM
Maths: 9AM
Physics: 11AM


In [8]:

# Replace the previous rule for high marks and low attendance
modified_rule4 = ctrl.Rule(
    marks['high'] & attendance['low'],
    performance['poor']
)

modified_ctrl = ctrl.ControlSystem([
    rule1, rule2, rule3,
    modified_rule4, rule5, rule6, rule7
])

modified_system = ctrl.ControlSystemSimulation(modified_ctrl)

# Test a student with high marks but low attendance
modified_system.input['attendance'] = 20
modified_system.input['marks'] = 90

modified_system.compute()

print("Attendance:", 20)
print("Marks:", 90)
print("Performance after rule change:",
      round(modified_system.output['performance'], 2))


Attendance: 20
Marks: 90
Performance after rule change: 14.86
